# Stateful Genetic Algorithm Phases: Exploration → Intensification → Exploitation

This notebook presents a **self-contained conceptual example** of how a genetic algorithm can change its behavior across explicit phases.

We will combine two ideas:

1. **Strategy** — genetic operators are interchangeable behaviors.
2. **State** — the algorithm can move through named phases, and each phase prefers a different operator profile.

The phase vocabulary used here is intentionally intuitive:

- **Exploration**: prefer diversity and larger jumps.
- **Intensification**: focus search around promising regions.
- **Exploitation**: apply smaller, more conservative refinements.

This notebook also introduces a small **runtime context** object to capture the signals that drive transitions, such as progress, stale fitness, and improvement ratio.

By the end, the notebook will show:

- a minimal optimization problem;
- simple interchangeable operators;
- explicit GA states;
- transition rules based on runtime metrics;
- a generation-by-generation trace explaining what the controller decided.

In [2]:
from __future__ import annotations

from dataclasses import dataclass, field
import random
from typing import Callable, Protocol

random.seed(7)

GENE_POOL = [1, 2, 3, 4, 5, 6]
TARGET = [1, 2, 3, 4, 5, 6]
Individual = list[int]
Population = list[Individual]

def clone_individual(individual: Individual) -> Individual:
    """Return a shallow copy of one candidate solution."""
    return list(individual)

def format_individual(individual: Individual) -> str:
    """Format one candidate for human-friendly notebook output."""
    return "[" + ", ".join(str(gene) for gene in individual) + "]"

def make_initial_population(size: int) -> Population:
    """Create a deterministic toy population of shuffled permutations."""
    population: Population = []
    for _ in range(size):
        candidate = list(GENE_POOL)
        random.shuffle(candidate)
        population.append(candidate)
    return population

def fitness(individual: Individual) -> float:
    """Return a minimization score for the toy permutation problem.

    The perfect answer is the sorted list `[1, 2, 3, 4, 5, 6]`.

    The score combines two pressures:
    1. position penalty: how far each gene is from its target value;
    2. adjacency penalty: how jagged the sequence is from one position to the next.

    Lower is better.
    """
    position_penalty = sum(abs(gene - TARGET[index]) for index, gene in enumerate(individual))
    adjacency_penalty = sum(abs(individual[index] - individual[index + 1]) for index in range(len(individual) - 1))
    return float(position_penalty * 3 + adjacency_penalty)

population = make_initial_population(size=6)

print("Initial toy population and fitness values:")
for index, candidate in enumerate(population, start=1):
    print(f"  {index}. {format_individual(candidate)} -> fitness={fitness(candidate):.1f}")

Initial toy population and fitness values:
  1. [5, 1, 6, 4, 2, 3] -> fitness=56.0
  2. [3, 4, 2, 6, 5, 1] -> fitness=48.0
  3. [3, 6, 4, 1, 2, 5] -> fitness=54.0
  4. [3, 6, 5, 2, 1, 4] -> fitness=59.0
  5. [2, 3, 4, 6, 5, 1] -> fitness=39.0
  6. [2, 3, 1, 4, 6, 5] -> fitness=27.0


## 1. A tiny optimization problem

The notebook uses a deliberately tiny problem: each individual is a **permutation of six integers**. The target arrangement is the ordered list:

$$[1, 2, 3, 4, 5, 6]$$

The fitness function is deterministic and easy to reason about. It combines two penalties:

1. **Position penalty** — how far each gene is from the value expected in that position.
2. **Adjacency penalty** — how irregular the sequence is from one element to the next.

Formally, if $x_i$ is the gene at position $i$ and $t_i$ is the target value for that position, then the score is:

$$
f(x) = 3 \cdot \sum_{i=1}^{6} \lvert x_i - t_i \rvert + \sum_{i=1}^{5} \lvert x_i - x_{i+1} \rvert
$$

This is a toy objective, but it is useful for teaching because:

- it is a **minimization** problem;
- it reacts visibly to local and global changes;
- we can understand improvements by just looking at the permutation.

The first code cell defined helper functions, created the initial population, and printed the starting fitness values so we have a baseline before introducing any GA logic.

In [3]:
best_initial = min(population, key=fitness)
worst_initial = max(population, key=fitness)

print("Best initial candidate :", format_individual(best_initial), f"-> {fitness(best_initial):.1f}")
print("Worst initial candidate:", format_individual(worst_initial), f"-> {fitness(worst_initial):.1f}")
print("Target sequence        :", format_individual(TARGET), f"-> {fitness(TARGET):.1f}")

Best initial candidate : [2, 3, 1, 4, 6, 5] -> 27.0
Worst initial candidate: [3, 6, 5, 2, 1, 4] -> 59.0
Target sequence        : [1, 2, 3, 4, 5, 6] -> 5.0


## 2. Local operator strategies

Now we define simplified genetic operators. This is where the **Strategy pattern** appears first.

Each operator type exposes a stable interface, while concrete implementations vary the search behavior:

- **Selection strategies** decide how strongly the algorithm favors good individuals.
- **Crossover strategies** decide how parents are recombined.
- **Mutation strategies** decide how disruptive variation should be.

We intentionally use toy operators with visibly different personalities:

- `RandomPairSelection` keeps pressure low and favors exploration.
- `TournamentSelection` applies moderate pressure.
- `ElitePairSelection` applies strong pressure and is suitable for late refinement.
- `AlternatingCrossover` mixes parents broadly.
- `PrefixSuffixCrossover` preserves more parent structure.
- `ShuffleWindowMutation` makes larger jumps.
- `SwapMutation` performs a medium local perturbation.
- `GreedyAdjacentMutation` searches for a small improving local change.

The important thing here is not that these are the best operators. The relevant aspect is that **operators are pluggable behaviors**, and that is exactly what makes later orchestration by state possible.

In [12]:
class ISelectionStrategy(Protocol):
    """Choose two parents from a population."""

    name: str

    def select_parents(self, population: Population, fitness_function: Callable[[Individual], float]) -> tuple[Individual, Individual]:
        """Return two parent individuals selected from the population."""
        ...


class ICrossoverStrategy(Protocol):
    """Combine two parents into one child."""

    name: str

    def crossover(self, parent1: Individual, parent2: Individual) -> Individual:
        """Return a child individual created by combining two parents."""
        ...


class IMutationStrategy(Protocol):
    """Mutate an offspring with a configurable probability."""

    name: str

    def mutate(self, individual: Individual, probability: float) -> Individual:
        """Return a mutated individual based on the given probability."""
        ...


class RandomPairSelection(ISelectionStrategy):
    """Low-pressure selection that samples parents uniformly at random."""

    name = "random-pair"

    def select_parents(self, population: Population, fitness_function: Callable[[Individual], float]) -> tuple[Individual, Individual]:
        choices = random.sample(population, 2)
        return clone_individual(choices[0]), clone_individual(choices[1])


class TournamentSelection(ISelectionStrategy):
    """Moderate-pressure selection that keeps the best of a small random tournament."""

    name = "tournament"

    def __init__(self, tournament_size: int = 3) -> None:
        self.tournament_size = tournament_size

    def _pick_one(self, population: Population, fitness_function: Callable[[Individual], float]) -> Individual:
        competitors = random.sample(population, self.tournament_size)
        winner = min(competitors, key=fitness_function)
        return clone_individual(winner)

    def select_parents(self, population: Population, fitness_function: Callable[[Individual], float]) -> tuple[Individual, Individual]:
        return self._pick_one(population, fitness_function), self._pick_one(population, fitness_function)


class ElitePairSelection(ISelectionStrategy):
    """High-pressure selection that always chooses the two best individuals."""

    name = "elite-pair"

    def select_parents(self, population: Population, fitness_function: Callable[[Individual], float]) -> tuple[Individual, Individual]:
        ranked = sorted(population, key=fitness_function)
        return clone_individual(ranked[0]), clone_individual(ranked[1])


class AlternatingCrossover(ICrossoverStrategy):
    """Broad recombination that alternates parent contributions."""

    name = "alternating"

    def crossover(self, parent1: Individual, parent2: Individual) -> Individual:
        child: Individual = []
        used: set[int] = set()
        for index in range(len(parent1)):
            preferred = parent1[index] if index % 2 == 0 else parent2[index]
            if preferred not in used:
                child.append(preferred)
                used.add(preferred)
        for gene in parent1 + parent2:
            if gene not in used:
                child.append(gene)
                used.add(gene)
        return child


class PrefixSuffixCrossover(ICrossoverStrategy):
    """More conservative recombination that preserves a prefix from one parent."""

    name = "prefix-suffix"

    def __init__(self, split_index: int = 3) -> None:
        self.split_index = split_index

    def crossover(self, parent1: Individual, parent2: Individual) -> Individual:
        child = list(parent1[: self.split_index])
        for gene in parent2:
            if gene not in child:
                child.append(gene)
        return child


class ShuffleWindowMutation(IMutationStrategy):
    """Heavy mutation that shuffles a local window and promotes exploration."""

    name = "shuffle-window"

    def __init__(self, window_size: int = 4) -> None:
        self.window_size = window_size

    def mutate(self, individual: Individual, probability: float) -> Individual:
        candidate = clone_individual(individual)
        if random.random() > probability:
            return candidate
        start = random.randint(0, len(candidate) - self.window_size)
        window = candidate[start : start + self.window_size]
        random.shuffle(window)
        candidate[start : start + self.window_size] = window
        return candidate


class SwapMutation(IMutationStrategy):
    """Medium mutation that swaps two random positions."""

    name = "swap"

    def mutate(self, individual: Individual, probability: float) -> Individual:
        candidate = clone_individual(individual)
        if random.random() > probability:
            return candidate
        i, j = sorted(random.sample(range(len(candidate)), 2))
        candidate[i], candidate[j] = candidate[j], candidate[i]
        return candidate


class GreedyAdjacentMutation(IMutationStrategy):
    """Light local search that keeps the best improving adjacent swap."""

    name = "greedy-adjacent"

    def mutate(self, individual: Individual, probability: float) -> Individual:
        candidate = clone_individual(individual)
        if random.random() > probability:
            return candidate
        best = clone_individual(candidate)
        best_score = fitness(best)
        for index in range(len(candidate) - 1):
            trial = clone_individual(candidate)
            trial[index], trial[index + 1] = trial[index + 1], trial[index]
            trial_score = fitness(trial)
            if trial_score < best_score:
                best = trial
                best_score = trial_score
        return best


toy_parent_a = clone_individual(population[0])
toy_parent_b = clone_individual(population[1])

print("Parent A:", format_individual(toy_parent_a))
print("Parent B:", format_individual(toy_parent_b))
print()
print("Alternating child :", format_individual(AlternatingCrossover().crossover(toy_parent_a, toy_parent_b)))
print("Prefix child      :", format_individual(PrefixSuffixCrossover(split_index=3).crossover(toy_parent_a, toy_parent_b)))
print()
print("Heavy mutation    :", format_individual(ShuffleWindowMutation().mutate(toy_parent_a, probability=1.0)))
print("Medium mutation   :", format_individual(SwapMutation().mutate(toy_parent_a, probability=1.0)))
print("Light mutation    :", format_individual(GreedyAdjacentMutation().mutate(toy_parent_a, probability=1.0)))

Parent A: [5, 1, 6, 4, 2, 3]
Parent B: [3, 4, 2, 6, 5, 1]

Alternating child : [5, 4, 6, 2, 1, 3]
Prefix child      : [5, 1, 6, 3, 4, 2]

Heavy mutation    : [5, 1, 2, 4, 3, 6]
Medium mutation   : [5, 4, 6, 1, 2, 3]
Light mutation    : [1, 5, 6, 4, 2, 3]


## 3. Runtime context and the operator bundle

Before we can talk about state transitions, the algorithm needs a compact way to describe what is happening in the current generation. That is the purpose of a **context object**.

The `GenerationContext` below captures the minimum runtime signals needed for policy decisions:

- `generation` and `max_generations`
- `progress_ratio`
- `best_fitness` and `previous_best_fitness`
- `stale_generations`
- `improvement_ratio`

The progress ratio is simply:

$$
\text{progress\_ratio} = \frac{\text{generation}}{\text{max\_generations}}
$$

And the improvement ratio is computed as:

$$
\text{improvement\_ratio} = \frac{\text{previous\_best} - \text{current\_best}}{\max\left(\left|\text{previous\_best}\right|, \varepsilon\right)}
$$

where $\varepsilon$ prevents division by zero.

We also introduce `GenerationOperators`, a tiny value object that groups the active selection, crossover, mutation, and mutation probability chosen for one generation. This is conceptually similar to a **Composite** or **bundle** object: instead of passing four separate decisions around, we package them together as one coherent configuration.

In [11]:
@dataclass
class GenerationContext:
    """Snapshot of the optimization loop at one generation."""

    generation: int
    max_generations: int
    best_fitness: float
    previous_best_fitness: float | None
    stale_generations: int
    elapsed_generations: int

    @property
    def progress_ratio(self) -> float:
        """Return the completed fraction of the GA run."""
        return self.generation / self.max_generations

    @property
    def improvement_ratio(self) -> float:
        """Return the relative improvement against the previous generation."""
        if self.previous_best_fitness is None:
            return 0.0
        baseline = max(abs(self.previous_best_fitness), 1e-9)
        raw_improvement = self.previous_best_fitness - self.best_fitness
        return max(0.0, raw_improvement / baseline)


@dataclass
class GenerationOperators:
    """Operators that are active for one generation."""

    selection: ISelectionStrategy
    crossover: ICrossoverStrategy
    mutation: IMutationStrategy
    mutation_probability: float


demo_context = GenerationContext(
    generation=4,
    max_generations=12,
    best_fitness=18.0,
    previous_best_fitness=20.0,
    stale_generations=0,
    elapsed_generations=4,
)

print("Demo context:")
print("  Progress ratio   :", f"{demo_context.progress_ratio:.1%}")
print("  Improvement ratio:", f"{demo_context.improvement_ratio:.1%}")

Demo context:
  Progress ratio   : 33.3%
  Improvement ratio: 10.0%


## 4. Transition conditions and explicit states

This is the heart of the notebook. We now model the GA phases explicitly with the **State pattern**.

Each state represents a named search posture:

- `ExplorationState` prefers diversity and uses disruptive mutation.
- `IntensificationState` becomes more selective and tries to consolidate promising structures.
- `ExploitationState` applies strong pressure and smaller local refinements.

A state has two responsibilities:

1. decide which operators should be active now;
2. expose the rules that determine when the controller should move to another state.

To keep transition logic modular, we also define a few small rule objects such as `ProgressAtLeast`, `StaleAtLeast`, and `ImprovementBelow`. This is a lightweight form of the **Specification pattern**: each condition answers one question, and transitions compose conditions instead of burying every rule inside a giant `if` block.

This separation is important because it keeps the example readable:

- operator choice belongs to the current state;
- transition criteria belong to reusable conditions;
- the controller later becomes responsible for asking the state what to do.

In [13]:
from abc import ABC, abstractmethod


class ICondition(Protocol):
    """Return whether a runtime condition is satisfied."""

    def matches(self, context: GenerationContext) -> bool:
        """Return `True` when the condition is satisfied."""
        ...


@dataclass
class ProgressAtLeast(ICondition):
    """Match once the run reaches a minimum progress ratio."""

    threshold: float

    def matches(self, context: GenerationContext) -> bool:
        return context.progress_ratio >= self.threshold


@dataclass
class StaleAtLeast(ICondition):
    """Match once the best fitness has stopped improving for long enough."""

    threshold: int

    def matches(self, context: GenerationContext) -> bool:
        return context.stale_generations >= self.threshold


@dataclass
class ImprovementBelow(ICondition):
    """Match when the relative improvement falls under a chosen threshold."""

    threshold: float

    def matches(self, context: GenerationContext) -> bool:
        return context.improvement_ratio < self.threshold


@dataclass
class TransitionRule:
    """Describe when the controller should jump from one state to another."""

    label: str
    target_state: Callable[[], "PhaseState"]
    conditions: list[ICondition]

    def matches(self, context: GenerationContext) -> bool:
        return all(condition.matches(context) for condition in self.conditions)


class PhaseState(ABC):
    """Base class for one explicit GA phase."""

    name = "phase"
    description = "generic search phase"

    @abstractmethod
    def resolve_operators(self, context: GenerationContext) -> GenerationOperators:
        """Return the operators that should govern the current generation."""
        ...

    @abstractmethod
    def transition_rules(self) -> list[TransitionRule]:
        """Return the ordered transition rules for this phase."""
        ...


class ExplorationState(PhaseState):
    """Early search phase that prioritizes diversity and larger jumps."""

    name = "exploration"
    description = "low pressure, broad crossover, heavy mutation"

    def resolve_operators(self, context: GenerationContext) -> GenerationOperators:
        return GenerationOperators(
            selection=RandomPairSelection(),
            crossover=AlternatingCrossover(),
            mutation=ShuffleWindowMutation(window_size=4),
            mutation_probability=0.95,
        )

    def transition_rules(self) -> list[TransitionRule]:
        return [
            TransitionRule(
                label="progress reached intensification zone",
                target_state=IntensificationState,
                conditions=[ProgressAtLeast(0.35)],
            )
        ]


class IntensificationState(PhaseState):
    """Middle phase that balances diversity and selective pressure."""

    name = "intensification"
    description = "moderate pressure, structure-preserving crossover, medium mutation"

    def resolve_operators(self, context: GenerationContext) -> GenerationOperators:
        return GenerationOperators(
            selection=TournamentSelection(tournament_size=3),
            crossover=PrefixSuffixCrossover(split_index=3),
            mutation=SwapMutation(),
            mutation_probability=0.55,
        )

    def transition_rules(self) -> list[TransitionRule]:
        return [
            TransitionRule(
                label="search is stale long enough",
                target_state=ExploitationState,
                conditions=[StaleAtLeast(2)],
            ),
            TransitionRule(
                label="late run refinement",
                target_state=ExploitationState,
                conditions=[ProgressAtLeast(0.75)],
            ),
        ]


class ExploitationState(PhaseState):
    """Late phase that focuses on conservative local improvement."""

    name = "exploitation"
    description = "high pressure, conservative crossover, light local mutation"

    def resolve_operators(self, context: GenerationContext) -> GenerationOperators:
        return GenerationOperators(
            selection=ElitePairSelection(),
            crossover=PrefixSuffixCrossover(split_index=4),
            mutation=GreedyAdjacentMutation(),
            mutation_probability=0.25,
        )

    def transition_rules(self) -> list[TransitionRule]:
        return [
            TransitionRule(
                label="late-phase stagnation fallback",
                target_state=IntensificationState,
                conditions=[ProgressAtLeast(0.85), StaleAtLeast(3), ImprovementBelow(0.01)],
            )
        ]


demo_state = ExplorationState()
demo_operators = demo_state.resolve_operators(demo_context)
print("Demo state name :", demo_state.name)
print("Selection       :", demo_operators.selection.name)
print("Crossover       :", demo_operators.crossover.name)
print("Mutation        :", demo_operators.mutation.name)
print("Mutation prob.  :", demo_operators.mutation_probability)

Demo state name : exploration
Selection       : random-pair
Crossover       : alternating
Mutation        : shuffle-window
Mutation prob.  : 0.95


## 5. The controller: where State and Strategy meet

A common mistake when introducing adaptive behavior is to push every rule directly into the GA loop. That usually produces a growing block of conditional logic such as:

- if progress is low, use one operator;
- else if stale is high, use another;
- else if improvement is tiny, reduce mutation;
- and so on.

That approach works for a while, then the loop becomes difficult to read and maintain.

The `GAStateController` below centralizes that decision-making. Its job is small but crucial:

1. keep track of the **current state**;
2. evaluate transition rules in order;
3. resolve the active operator bundle for the generation.

This is the point where the patterns connect:

- **State** decides phase-specific behavior;
- **Strategy** supplies the interchangeable operators;
- **Specification** keeps the transition conditions modular;
- **Context Object** provides the runtime data needed for decisions.

After the controller, the simplified GA engine becomes straightforward: evaluate population, build context, ask the controller what to do, generate offspring, repeat.

In [16]:
@dataclass
class GenerationRecord:
    """Structured record of one generation for post-run analysis."""

    generation: int
    state_name: str
    transition_label: str | None
    best_fitness: float
    stale_generations: int
    improvement_ratio: float
    selection_name: str
    crossover_name: str
    mutation_name: str
    mutation_probability: float
    best_individual: Individual = field(default_factory=list)


class GAStateController:
    """Track the current GA phase and resolve operator changes over time."""

    def __init__(self, initial_state: PhaseState) -> None:
        self._state = initial_state

    @property
    def state_name(self) -> str:
        """Expose the current phase name for trace output."""
        return self._state.name

    def resolve(self, context: GenerationContext) -> tuple[GenerationOperators, str | None]:
        """Apply transitions if necessary and return the active operators."""
        transition_label: str | None = None
        for rule in self._state.transition_rules():
            if rule.matches(context):
                self._state = rule.target_state()
                transition_label = rule.label
                break
        return self._state.resolve_operators(context), transition_label


class SimpleStatefulGA:
    """Minimal GA engine used only to illustrate state-driven operator switching."""

    def __init__(self, population_size: int, max_generations: int, controller: GAStateController) -> None:
        self.population_size = population_size
        self.max_generations = max_generations
        self.controller = controller

    def run(self, initial_population: Population) -> list[GenerationRecord]:
        """Execute the simplified GA and return a full generation trace."""
        population = [clone_individual(candidate) for candidate in initial_population]
        history: list[GenerationRecord] = []
        previous_best: float | None = None
        stale_generations = 0

        for generation in range(1, self.max_generations + 1):
            ranked = sorted(population, key=fitness)
            best = ranked[0]
            best_score = fitness(best)

            if previous_best is None or best_score < previous_best:
                stale_generations = 0
            else:
                stale_generations += 1

            context = GenerationContext(
                generation=generation,
                max_generations=self.max_generations,
                best_fitness=best_score,
                previous_best_fitness=previous_best,
                stale_generations=stale_generations,
                elapsed_generations=generation,
            )
            operators, transition_label = self.controller.resolve(context)

            record = GenerationRecord(
                generation=generation,
                state_name=self.controller.state_name,
                transition_label=transition_label,
                best_fitness=best_score,
                stale_generations=stale_generations,
                improvement_ratio=context.improvement_ratio,
                selection_name=operators.selection.name,
                crossover_name=operators.crossover.name,
                mutation_name=operators.mutation.name,
                mutation_probability=operators.mutation_probability,
                best_individual=clone_individual(best),
            )
            history.append(record)

            if transition_label is not None:
                print(f"         transition -> {transition_label}")
            print(
                f"Gen {generation:02d} | state={record.state_name:<14} | best={record.best_fitness:5.1f} | "
                f"stale={record.stale_generations} | improve={record.improvement_ratio:6.1%} | "
                f"sel={record.selection_name:<11} | cross={record.crossover_name:<13} | "
                f"mut={record.mutation_name:<15} | p={record.mutation_probability:.2f}"
            )

            new_population = [clone_individual(best)]
            while len(new_population) < self.population_size:
                parent1, parent2 = operators.selection.select_parents(ranked, fitness)
                child = operators.crossover.crossover(parent1, parent2)
                child = operators.mutation.mutate(child, operators.mutation_probability)
                new_population.append(child)

            population = new_population
            previous_best = best_score

        return history

## 6. Run the experiment and inspect the trace

The next cell executes the simplified GA using the explicit phase controller. A few details are worth watching while the trace prints:

- **When does the controller leave exploration?**
- **Does stale fitness accelerate the move toward exploitation?**
- **How do the active operators change with the phase?**
- **How does the mutation probability shrink as the search becomes more conservative?**

This trace is intentionally verbose because the notebook is meant to be instructional.

In [17]:
random.seed(21)
experiment_population = make_initial_population(size=6)
controller = GAStateController(initial_state=ExplorationState())
experiment = SimpleStatefulGA(population_size=6, max_generations=12, controller=controller)
history = experiment.run(experiment_population)

Gen 01 | state=exploration    | best= 39.0 | stale=0 | improve=  0.0% | sel=random-pair | cross=alternating   | mut=shuffle-window  | p=0.95
Gen 02 | state=exploration    | best= 35.0 | stale=0 | improve= 10.3% | sel=random-pair | cross=alternating   | mut=shuffle-window  | p=0.95
Gen 03 | state=exploration    | best= 28.0 | stale=0 | improve= 20.0% | sel=random-pair | cross=alternating   | mut=shuffle-window  | p=0.95
Gen 04 | state=exploration    | best= 26.0 | stale=0 | improve=  7.1% | sel=random-pair | cross=alternating   | mut=shuffle-window  | p=0.95
         transition -> progress reached intensification zone
Gen 05 | state=intensification | best= 20.0 | stale=0 | improve= 23.1% | sel=tournament  | cross=prefix-suffix | mut=swap            | p=0.55
Gen 06 | state=intensification | best= 20.0 | stale=1 | improve=  0.0% | sel=tournament  | cross=prefix-suffix | mut=swap            | p=0.55
         transition -> search is stale long enough
Gen 07 | state=exploitation   | best= 20

## 7. Summarize what happened

The trace above is useful while reading the run sequentially, but a compact summary helps us interpret the result after the fact.

The next code cell extracts three views from the generation history:

1. the sequence of active states;
2. the generations where transitions occurred;
3. the best fitness progression.

This makes it easier to verify whether the controller actually influenced the search behavior instead of merely existing as decorative architecture.

In [18]:
state_sequence = [record.state_name for record in history]
transition_events = [
    (record.generation, record.transition_label, record.state_name)
    for record in history
    if record.transition_label is not None
]
fitness_progression = [record.best_fitness for record in history]

print("State sequence:")
print("  ", " -> ".join(state_sequence))

print("\nTransitions:")
for generation, label, state_name in transition_events:
    print(f"  Generation {generation:02d}: {label} -> {state_name}")

print("\nBest fitness by generation:")
for record in history:
    marker = "*" if record.transition_label else " "
    print(f"{marker} Gen {record.generation:02d}: best={record.best_fitness:5.1f} | state={record.state_name:<14} | individual={format_individual(record.best_individual)}")

State sequence:
   exploration -> exploration -> exploration -> exploration -> intensification -> intensification -> exploitation -> exploitation -> exploitation -> exploitation -> exploitation -> exploitation

Transitions:
  Generation 05: progress reached intensification zone -> intensification
  Generation 07: search is stale long enough -> exploitation

Best fitness by generation:
  Gen 01: best= 39.0 | state=exploration    | individual=[4, 3, 1, 2, 6, 5]
  Gen 02: best= 35.0 | state=exploration    | individual=[1, 3, 6, 4, 5, 2]
  Gen 03: best= 28.0 | state=exploration    | individual=[1, 3, 2, 6, 4, 5]
  Gen 04: best= 26.0 | state=exploration    | individual=[3, 2, 1, 4, 6, 5]
* Gen 05: best= 20.0 | state=intensification | individual=[1, 3, 2, 4, 6, 5]
  Gen 06: best= 20.0 | state=intensification | individual=[1, 3, 2, 4, 6, 5]
* Gen 07: best= 20.0 | state=exploitation   | individual=[1, 3, 2, 4, 6, 5]
  Gen 08: best= 20.0 | state=exploitation   | individual=[1, 3, 2, 4, 6, 5]
  

## 8. Design pattern analysis

This example intentionally layers several small patterns together. Their roles are different, and that distinction matters.

### Strategy
The operator classes are classic strategy objects. Each one encapsulates a replaceable behavior behind a stable interface:

- selection strategy;
- crossover strategy;
- mutation strategy.

Without Strategy, every operator choice would be buried inside the GA loop as conditional branches.

### State
The explicit phases are state objects. A state does not execute the entire GA; instead, it answers the question: **given the current phase, which operator profile makes sense right now?**

That is the key conceptual difference:

- **Strategy** = interchangeable operator behavior;
- **State** = phase-dependent coordination of strategies.

### Context Object
`GenerationContext` packages runtime signals into one object. This keeps method signatures short and prevents decision logic from depending on a long list of loose parameters.

### Specification
The small condition classes (`ProgressAtLeast`, `StaleAtLeast`, `ImprovementBelow`) isolate the semantics of each rule. They make the transition logic readable, testable, and composable.

### Composite / Bundle
`GenerationOperators` groups the active decisions for a generation into one value object. It is a lightweight composite: one object now represents the behavioral configuration of the generation.

### Why this combination is useful
This pattern combination prevents the optimizer loop from turning into a long conditional script. The loop keeps responsibility for execution, while the controller and states concentrate the adaptation logic. That is typically the sweet spot for maintainability.

## 9. How this concept maps back to a real codebase

If this idea were later moved into a production architecture, the conceptual pieces would map naturally to layered components:

- the **operator interfaces** would remain domain-level contracts;
- the **phase controller or policy** would likely become an orchestrating domain or application abstraction;
- the **concrete states and transition rules** would live in infrastructure if they depend on configuration or implementation-specific operators;
- the **optimizer loop** would remain the execution engine that evaluates populations and applies the active operator bundle.

In other words, the loop should keep control of execution, while the adaptive layer controls **which behaviors become active**.

That separation is the main architectural lesson from this notebook. It gives you a path to add adaptive GA behavior without rewriting every existing operator or turning the optimizer into a giant rule switchboard.